# Stochastic Processes with RustQuant (Python)

Python version of `03_stochastic_processes.ipynb` using the PyO3 bindings.

## Setup

In [ ]:
import math
from RustQuant.stochastics import (
    GeometricBrownianMotion, ArithmeticBrownianMotion, BrownianMotion,
    OrnsteinUhlenbeck, CoxIngersollRoss, HullWhite, ExtendedVasicek,
    HoLee, BlackDermanToy, MertonJumpDiffusion,
)

## 1. Geometric Brownian Motion

$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$

In [ ]:
gbm = GeometricBrownianMotion(mu=0.05, sigma=0.20)
out = gbm.simulate(x0=100.0, t_end=1.0, n_steps=252, n_paths=5)

print(f"GBM: {len(out.paths)} paths, {len(out.times)} time steps")
for i, path in enumerate(out.paths):
    print(f"  Path {i+1}: start = {path[0]:.2f}, end = {path[-1]:.2f}")

## 2. Arithmetic Brownian Motion

$dX_t = \mu \, dt + \sigma \, dW_t$

In [ ]:
abm = ArithmeticBrownianMotion(mu=0.05, sigma=0.20)
out = abm.simulate(x0=100.0, t_end=1.0, n_steps=252, n_paths=3)
for i, path in enumerate(out.paths):
    print(f"  Path {i+1}: start = {path[0]:.2f}, end = {path[-1]:.2f}")

## 3. Ornstein-Uhlenbeck (Mean-Reverting)

$dX_t = \theta(\mu - X_t) \, dt + \sigma \, dW_t$

In [ ]:
ou = OrnsteinUhlenbeck(mu=0.5, sigma=0.1, theta=2.0)
out = ou.simulate(x0=0.5, t_end=5.0, n_steps=1000, n_paths=3)

print("OU (mean-reverting to 0.5):")
for i, path in enumerate(out.paths):
    avg = sum(path) / len(path)
    print(f"  Path {i+1}: mean = {avg:.4f}, final = {path[-1]:.4f}")

## 4. Cox-Ingersoll-Ross (Non-Negative Mean-Reverting)

$dr_t = \theta(\mu - r_t) \, dt + \sigma \sqrt{r_t} \, dW_t$

In [ ]:
cir = CoxIngersollRoss(mu=0.05, sigma=0.1, theta=0.9)
out = cir.simulate(x0=0.03, t_end=10.0, n_steps=2520, n_paths=3)

print("CIR (mean-reverting to 0.05):")
for i, path in enumerate(out.paths):
    print(f"  Path {i+1}: final = {path[-1]:.4f}, min = {min(path):.4f}, max = {max(path):.4f}")

## 5. Hull-White Short-Rate Model

$dr_t = (\theta - \alpha r_t) \, dt + \sigma \, dW_t$

In [ ]:
hw = HullWhite(alpha=0.1, sigma=0.01, theta=0.2)
out = hw.simulate(x0=0.03, t_end=10.0, n_steps=2520, n_paths=3)

print("Hull-White:")
for i, path in enumerate(out.paths):
    print(f"  Path {i+1}: start = {path[0]:.4f}, end = {path[-1]:.4f}")

## 6. Merton Jump Diffusion

GBM with Poisson jumps for sudden market moves.

In [ ]:
mjd = MertonJumpDiffusion(mu=0.05, sigma=0.20, lambda_=5.0, jump_mean=-0.02, jump_volatility=0.10)
out = mjd.simulate(x0=100.0, t_end=1.0, n_steps=252, n_paths=3)

print("Merton Jump Diffusion:")
for i, path in enumerate(out.paths):
    print(f"  Path {i+1}: start = {path[0]:.2f}, end = {path[-1]:.2f}")

## 7. Monte Carlo Statistics (10,000 GBM Paths)

In [ ]:
gbm = GeometricBrownianMotion(mu=0.05, sigma=0.20)
out = gbm.simulate(x0=100.0, t_end=1.0, n_steps=252, n_paths=10_000, parallel=True)

terminals = [p[-1] for p in out.paths]
mean_t = sum(terminals) / len(terminals)
var_t = sum((x - mean_t)**2 for x in terminals) / len(terminals)

print(f"GBM Monte Carlo ({len(terminals)} paths):")
print(f"  Mean terminal value:  {mean_t:.2f}")
print(f"  Std dev:              {math.sqrt(var_t):.2f}")
print(f"  Expected (analytic):  {100.0 * math.exp(0.05):.2f}")

## Available Processes

| Process | Python Class | Parameters |
|---------|-------------|------------|
| Standard BM | `BrownianMotion()` | - |
| Arithmetic BM | `ArithmeticBrownianMotion(mu, sigma)` | drift, vol |
| Geometric BM | `GeometricBrownianMotion(mu, sigma)` | drift, vol |
| Ornstein-Uhlenbeck | `OrnsteinUhlenbeck(mu, sigma, theta)` | mean, vol, speed |
| Cox-Ingersoll-Ross | `CoxIngersollRoss(mu, sigma, theta)` | mean, vol, speed |
| Hull-White | `HullWhite(alpha, sigma, theta)` | reversion, vol, drift |
| Extended Vasicek | `ExtendedVasicek(alpha, sigma, theta)` | reversion, vol, drift |
| Ho-Lee | `HoLee(sigma, theta)` | vol, drift |
| Black-Derman-Toy | `BlackDermanToy(sigma, theta)` | vol, drift |
| Merton Jump Diffusion | `MertonJumpDiffusion(...)` | drift, vol, intensity, jump params |